# SXO Page-Query Dynamics

Understand how top pages, rankings, and queries shift in a **single recent window** with an SXO lens.

## What this notebook helps answer
- Which top pages moved most in the last `X` weeks?
- Which queries likely drove those page-level changes?
- Do those changes look more like demand shifts, discoverability shifts, or SERP competitiveness shifts?

## How to use
1. Run **Setup (run once)**.
2. Set values in **Parameters** (especially `RECENT_WEEKS` and `INCLUDE_HOMEPAGE`).
3. Run all cells from top to bottom.
4. Use the interpretation/checklist at the end to decide action.

This notebook focuses on practical diagnosis, not traffic reporting.

In [1]:
#@title Setup (run once)
import os
import sys
from datetime import date, timedelta

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidanm-lla/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q "plotly>=6.1.1" "kaleido>=1.2.0"
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import build_date_params, get_client, load_sql_template, run_query

lifeline_theme.inject_fonts()
client = get_client()

In [2]:
#@title Parameters
TOP_N_PAGES = 25 #@param {type:"integer"}
RECENT_WEEKS = 8 #@param {type:"integer"}
INCLUDE_HOMEPAGE = False #@param {type:"boolean"}
MIN_IMPRESSIONS_WEEKLY = 0 #@param {type:"integer"}
MIN_QUERY_IMPRESSIONS = 0 #@param {type:"integer"}
FOCUS_PAGE_COUNT = 15 #@param {type:"integer"}

if RECENT_WEEKS < 2:
    raise ValueError("RECENT_WEEKS must be at least 2 so comparisons are meaningful.")

recent_days = RECENT_WEEKS * 7
comparison_days = recent_days * 2
analysis_end_date = date.today() - timedelta(days=2)

# Pull only what we need: recent window + prior comparison window.
analysis_start_date = analysis_end_date - timedelta(days=comparison_days - 1)

print(f"Project: {config.PROJECT_ID}")
print(f"Search Console dataset: {config.SEARCHCONSOLE_DATASET}")
print(f"Analysis window: {analysis_start_date} to {analysis_end_date} ({comparison_days} days)")
print(f"Recent focus window: last {RECENT_WEEKS} week(s) ({recent_days} days)")
print(f"Homepage included: {INCLUDE_HOMEPAGE}")

Project: lifeline-website-480522
Search Console dataset: searchconsole
Analysis window: 2025-11-12 to 2026-03-03 (112 days)
Recent focus window: last 8 week(s) (56 days)
Homepage included: False


In [3]:
# Freshness / sanity check
freshness_sql = f"""
SELECT
  (SELECT MAX(report_date) FROM `{config.PROJECT_ID}.{config.SEARCHCONSOLE_DATASET}.seo_page_daily`) AS max_page_date,
  (SELECT MAX(report_date) FROM `{config.PROJECT_ID}.{config.SEARCHCONSOLE_DATASET}.curated_search_query_page_daily`) AS max_query_date
"""

freshness = run_query(client, freshness_sql)
max_page_date = pd.to_datetime(freshness.loc[0, "max_page_date"]).date()
max_query_date = pd.to_datetime(freshness.loc[0, "max_query_date"]).date()
effective_end_date = min(analysis_end_date, max_page_date, max_query_date)

print(f"Latest page-level date: {max_page_date}")
print(f"Latest query-level date: {max_query_date}")
print(f"Effective analysis end date: {effective_end_date}")

if effective_end_date <= analysis_start_date:
    raise ValueError("No usable data in configured window. Adjust dates or check data freshness.")

availability_sql = f"""
WITH page_weekly AS (
  SELECT
    DATE_TRUNC(report_date, WEEK(MONDAY)) AS week_start,
    SUM(gsc_clicks) AS clicks
  FROM `{config.PROJECT_ID}.{config.SEARCHCONSOLE_DATASET}.seo_page_daily`
  WHERE report_date BETWEEN DATE(@start_date) AND DATE(@end_date)
  GROUP BY week_start
),
query_weekly AS (
  SELECT
    DATE_TRUNC(report_date, WEEK(MONDAY)) AS week_start,
    SUM(clicks) AS clicks
  FROM `{config.PROJECT_ID}.{config.SEARCHCONSOLE_DATASET}.curated_search_query_page_daily`
  WHERE report_date BETWEEN DATE(@start_date) AND DATE(@end_date)
  GROUP BY week_start
)
SELECT
  (SELECT COUNTIF(clicks > 0) FROM page_weekly) AS page_nonzero_weeks,
  (SELECT COUNTIF(clicks > 0) FROM query_weekly) AS query_nonzero_weeks
"""

availability = run_query(
    client,
    availability_sql,
    params=[
        bigquery.ScalarQueryParameter("start_date", "DATE", analysis_start_date),
        bigquery.ScalarQueryParameter("end_date", "DATE", effective_end_date),
    ],
)

page_nonzero_weeks = int(availability.loc[0, "page_nonzero_weeks"])
query_nonzero_weeks = int(availability.loc[0, "query_nonzero_weeks"])

print(f"Weeks with non-zero page clicks in analysis window: {page_nonzero_weeks}")
print(f"Weeks with non-zero query clicks in analysis window: {query_nonzero_weeks}")

if page_nonzero_weeks < RECENT_WEEKS or query_nonzero_weeks < RECENT_WEEKS:
    print(
        "Warning: fewer non-zero GSC weeks are available than RECENT_WEEKS. "
        "Leading weeks may appear as zero until data backfill catches up."
    )

Latest page-level date: 2026-03-03
Latest query-level date: 2026-03-02
Effective analysis end date: 2026-03-02


In [4]:
# Load weekly page dynamics for top pages
page_sql = load_sql_template(
    "sql/notebooks/seo_sxo_page_weekly_dynamics.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
)

page_params = [
    bigquery.ScalarQueryParameter("start_date", "DATE", analysis_start_date),
    bigquery.ScalarQueryParameter("end_date", "DATE", effective_end_date),
    bigquery.ScalarQueryParameter("top_n_pages", "INT64", TOP_N_PAGES),
    bigquery.ScalarQueryParameter("include_homepage", "BOOL", INCLUDE_HOMEPAGE),
    bigquery.ScalarQueryParameter("min_impressions_weekly", "INT64", MIN_IMPRESSIONS_WEEKLY),
]

df_page_weekly = run_query(client, page_sql, params=page_params)
if df_page_weekly.empty:
    raise ValueError("No weekly page data returned. Try lowering thresholds or widening date windows.")

df_page_weekly["week_start"] = pd.to_datetime(df_page_weekly["week_start"])

print(f"Rows returned: {len(df_page_weekly):,}")
df_page_weekly.head()

Rows returned: 425


,week_start,page_path,clicks,impressions,ctr,avg_position
0,2025-11-10,/131114,0,0,0.0,NaN
1,2025-11-10,/about-us/our-services,0,0,0.0,NaN
2,2025-11-10,/about/careers,0,0,0.0,NaN
3,2025-11-10,/about/contact-us,0,0,0.0,NaN
4,2025-11-10,/about/find-a-lifeline-centre-near-you,0,0,0.0,NaN


In [5]:
# Derive recent movers from weekly data
latest_week = df_page_weekly["week_start"].max()
recent_week_cutoff = latest_week - pd.Timedelta(days=(RECENT_WEEKS - 1) * 7)
prior_week_cutoff = recent_week_cutoff - pd.Timedelta(days=RECENT_WEEKS * 7)

recent = df_page_weekly[df_page_weekly["week_start"] >= recent_week_cutoff].copy()
prior = df_page_weekly[
    (df_page_weekly["week_start"] >= prior_week_cutoff)
    & (df_page_weekly["week_start"] < recent_week_cutoff)
].copy()

recent_summary = (
    recent.groupby("page_path", as_index=False)
    .agg(
        recent_clicks=("clicks", "sum"),
        recent_impressions=("impressions", "sum"),
    )
)
recent_summary["recent_ctr"] = recent_summary["recent_clicks"] / recent_summary["recent_impressions"].replace(0, pd.NA)

prior_summary = (
    prior.groupby("page_path", as_index=False)
    .agg(
        prior_clicks=("clicks", "sum"),
        prior_impressions=("impressions", "sum"),
    )
)
prior_summary["prior_ctr"] = prior_summary["prior_clicks"] / prior_summary["prior_impressions"].replace(0, pd.NA)

recent["position_x_impressions"] = recent["avg_position"] * recent["impressions"]
prior["position_x_impressions"] = prior["avg_position"] * prior["impressions"]

recent_position = (
    recent.groupby("page_path", as_index=False)
    .agg(
        recent_position_weighted=("position_x_impressions", "sum"),
        recent_impressions_for_position=("impressions", "sum"),
    )
)
recent_position["recent_avg_position"] = (
    recent_position["recent_position_weighted"]
    / recent_position["recent_impressions_for_position"].replace(0, pd.NA)
)
recent_position = recent_position[["page_path", "recent_avg_position"]]

prior_position = (
    prior.groupby("page_path", as_index=False)
    .agg(
        prior_position_weighted=("position_x_impressions", "sum"),
        prior_impressions_for_position=("impressions", "sum"),
    )
)
prior_position["prior_avg_position"] = (
    prior_position["prior_position_weighted"]
    / prior_position["prior_impressions_for_position"].replace(0, pd.NA)
)
prior_position = prior_position[["page_path", "prior_avg_position"]]

page_movers = recent_summary.merge(prior_summary, on="page_path", how="left")
page_movers = page_movers.merge(recent_position, on="page_path", how="left")
page_movers = page_movers.merge(prior_position, on="page_path", how="left")

for col in ["prior_clicks", "prior_impressions", "prior_ctr", "prior_avg_position"]:
    page_movers[col] = page_movers[col].fillna(0)

page_movers["click_delta"] = page_movers["recent_clicks"] - page_movers["prior_clicks"]
page_movers["impression_delta"] = page_movers["recent_impressions"] - page_movers["prior_impressions"]
page_movers["ctr_delta"] = page_movers["recent_ctr"] - page_movers["prior_ctr"]
page_movers["position_delta"] = page_movers["recent_avg_position"] - page_movers["prior_avg_position"]

page_movers = page_movers.sort_values("click_delta", ascending=False)
page_movers.head(15)

,page_path,recent_clicks,recent_impressions,recent_ctr,prior_clicks,prior_impressions,prior_ctr,recent_avg_position,prior_avg_position,click_delta,impression_delta,ctr_delta,position_delta
8,/get-help/national-services/national-alcohol-o...,6686,500932,0.013347,0,0,0.0,3.242528,0.0,6686,500932,0.013347,3.242528
5,/chat,1375,257980,0.00533,0,0,0.0,11.901434,0.0,1375,257980,0.00533,11.901434
21,/get-help/support-toolkit/topics/self-harm,1285,58829,0.021843,0,0,0.0,6.76962,0.0,1285,58829,0.021843,6.76962
23,/get-involved/volunteer/volunteer-as-a-crisis-...,713,16314,0.043705,0,0,0.0,4.339402,0.0,713,16314,0.043705,4.339402
12,/get-help/support-toolkit/techniques-and-guide...,479,72188,0.006635,0,0,0.0,5.593547,0.0,479,72188,0.006635,5.593547
14,/get-help/support-toolkit/techniques-and-guide...,433,95366,0.00454,0,0,0.0,5.726066,0.0,433,95366,0.00454,5.726066
0,/131114,431,131033,0.003289,0,0,0.0,3.677997,0.0,431,131033,0.003289,3.677997
19,/get-help/support-toolkit/tools-and-apps/i-am-...,424,37602,0.011276,0,0,0.0,7.167092,0.0,424,37602,0.011276,7.167092
9,/get-help/support-toolkit/safety-planning/beyo...,408,6155,0.066288,0,0,0.0,9.188465,0.0,408,6155,0.066288,9.188465
2,/about/careers,396,2704,0.14645,0,0,0.0,6.22966,0.0,396,2704,0.14645,6.22966


## Chart 1: Weekly clicks for top movers

This chart shows weekly click trend lines for the pages with the strongest recent momentum.

Why this helps:
- Quickly spot whether growth/decline is broad, sudden, or concentrated in a few pages.
- Separate stable trend movement from one-off spikes before taking action.

How to read it:
- Each line is one page.
- X-axis is week start date.
- The chart is limited to the configured `RECENT_WEEKS` window.
- Flat zero points can appear when a page had no measurable impressions/clicks in that week.

In [6]:
# Weekly trends for pages with strongest recent momentum
focus_pages = page_movers.head(FOCUS_PAGE_COUNT)["page_path"].tolist()

df_focus = df_page_weekly[
    (df_page_weekly["page_path"].isin(focus_pages))
    & (df_page_weekly["week_start"] >= recent_week_cutoff)
    & (df_page_weekly["week_start"] <= latest_week)
].copy()

# Ensure the chart always shows the full recent-week window, even when
# a page has no row in a given week (for example, filtered by low impressions).
recent_weeks = pd.date_range(
    start=recent_week_cutoff.normalize(),
    end=latest_week.normalize(),
    freq="W-MON",
)

focus_grid = pd.MultiIndex.from_product(
    [focus_pages, recent_weeks],
    names=["page_path", "week_start"],
).to_frame(index=False)

df_focus = focus_grid.merge(
    df_focus[["page_path", "week_start", "clicks"]],
    on=["page_path", "week_start"],
    how="left",
)

df_focus["clicks"] = df_focus["clicks"].fillna(0)
df_focus = df_focus.sort_values(["page_path", "week_start"])

fig_trend = px.line(
    df_focus,
    x="week_start",
    y="clicks",
    color="page_path",
    markers=True,
    template="lifeline",
    title=f"Weekly clicks for top recent movers (last {RECENT_WEEKS} weeks focus)",
)
fig_trend.update_xaxes(range=[recent_weeks.min(), recent_weeks.max()])
lifeline_theme.add_lifeline_logo(fig_trend)
fig_trend.show()

print(
    f"Trend chart coverage: {recent_weeks.min().date()} to {recent_weeks.max().date()} "
    f"({len(recent_weeks)} weeks)"
)

Trend chart coverage: 2026-01-12 to 2026-03-02 (8 weeks)


## Chart 2: Ranking vs CTR snapshot

This scatterplot compares ranking quality and click-through performance in the recent window.

Why this helps:
- Find pages with decent rankings but weak CTR (snippet/message opportunity).
- Find pages with strong CTR but weaker rankings (SEO discoverability opportunity).

How to read it:
- Further left = better average position.
- Higher = better CTR.
- Bigger bubbles = higher impressions.
- Color shows click change vs prior window (`click_delta`).

In [7]:
# Ranking vs demand snapshot (recent window)
fig_scatter = px.scatter(
    page_movers,
    x="recent_avg_position",
    y="recent_ctr",
    size="recent_impressions",
    color="click_delta",
    hover_data=["page_path", "recent_clicks", "prior_clicks", "impression_delta", "position_delta"],
    template="lifeline",
    title="Recent page performance: ranking vs CTR, sized by impressions",
)
fig_scatter.update_xaxes(autorange="reversed", title="Average position (lower is better)")
fig_scatter.update_yaxes(title="CTR")
lifeline_theme.add_lifeline_logo(fig_scatter)
fig_scatter.show()

## Table 1: Rising and falling pages

These tables list the biggest winners and losers by click change.

Why this helps:
- Prioritize which pages need investigation first.
- Compare whether movement comes from clicks, impressions, ranking, or CTR.

Tip: Start with the largest negative `click_delta` rows to triage risk quickly.

In [8]:
# Recent movers table (positive and negative)
cols = [
    "page_path",
    "recent_clicks",
    "prior_clicks",
    "click_delta",
    "recent_impressions",
    "prior_impressions",
    "impression_delta",
    "recent_avg_position",
    "prior_avg_position",
    "position_delta",
    "recent_ctr",
    "prior_ctr",
    "ctr_delta",
]

print("Top rising pages by click delta")
display(page_movers[cols].head(15))

print("Top falling pages by click delta")
display(page_movers[cols].sort_values("click_delta").head(15))

Top rising pages by click delta


,page_path,recent_clicks,prior_clicks,click_delta,recent_impressions,prior_impressions,impression_delta,recent_avg_position,prior_avg_position,position_delta,recent_ctr,prior_ctr,ctr_delta
8,/get-help/national-services/national-alcohol-o...,6686,0,6686,500932,0,500932,3.242528,0.0,3.242528,0.013347,0.0,0.013347
5,/chat,1375,0,1375,257980,0,257980,11.901434,0.0,11.901434,0.00533,0.0,0.00533
21,/get-help/support-toolkit/topics/self-harm,1285,0,1285,58829,0,58829,6.76962,0.0,6.76962,0.021843,0.0,0.021843
23,/get-involved/volunteer/volunteer-as-a-crisis-...,713,0,713,16314,0,16314,4.339402,0.0,4.339402,0.043705,0.0,0.043705
12,/get-help/support-toolkit/techniques-and-guide...,479,0,479,72188,0,72188,5.593547,0.0,5.593547,0.006635,0.0,0.006635
14,/get-help/support-toolkit/techniques-and-guide...,433,0,433,95366,0,95366,5.726066,0.0,5.726066,0.00454,0.0,0.00454
0,/131114,431,0,431,131033,0,131033,3.677997,0.0,3.677997,0.003289,0.0,0.003289
19,/get-help/support-toolkit/tools-and-apps/i-am-...,424,0,424,37602,0,37602,7.167092,0.0,7.167092,0.011276,0.0,0.011276
9,/get-help/support-toolkit/safety-planning/beyo...,408,0,408,6155,0,6155,9.188465,0.0,9.188465,0.066288,0.0,0.066288
2,/about/careers,396,0,396,2704,0,2704,6.22966,0.0,6.22966,0.14645,0.0,0.14645


Top falling pages by click delta


,page_path,recent_clicks,prior_clicks,click_delta,recent_impressions,prior_impressions,impression_delta,recent_avg_position,prior_avg_position,position_delta,recent_ctr,prior_ctr,ctr_delta
16,/get-help/support-toolkit/techniques-and-guide...,201,0,201,74445,0,74445,8.327315,0.0,8.327315,0.0027,0.0,0.0027
3,/about/contact-us,234,0,234,39921,0,39921,4.721275,0.0,4.721275,0.005862,0.0,0.005862
24,/text,243,0,243,37440,0,37440,6.969177,0.0,6.969177,0.00649,0.0,0.00649
13,/get-help/support-toolkit/techniques-and-guide...,246,0,246,26139,0,26139,5.576533,0.0,5.576533,0.009411,0.0,0.009411
15,/get-help/support-toolkit/techniques-and-guide...,273,0,273,121205,0,121205,6.295062,0.0,6.295062,0.002252,0.0,0.002252
22,/get-involved/donate/donate-to-lifeline,279,0,279,25346,0,25346,11.563718,0.0,11.563718,0.011008,0.0,0.011008
17,/get-help/support-toolkit/tools-and-apps/beyon...,282,0,282,2144,0,2144,5.792444,0.0,5.792444,0.13153,0.0,0.13153
11,/get-help/support-toolkit/techniques-and-guide...,307,0,307,47825,0,47825,6.407444,0.0,6.407444,0.006419,0.0,0.006419
7,/get-help/national-services/kids-helpline,321,0,321,30966,0,30966,5.307692,0.0,5.307692,0.010366,0.0,0.010366
10,/get-help/support-toolkit/techniques-and-guide...,327,0,327,32154,0,32154,6.388381,0.0,6.388381,0.01017,0.0,0.01017


## Query-level drilldown input

Next we load query-level changes for the same analysis window, so we can explain *why* page performance moved.

This bridges page-level symptoms (what changed) to query-level causes (what drove it).

In [9]:
# Load query-level recent changes for top pages
query_sql = load_sql_template(
    "sql/notebooks/seo_sxo_query_page_recent_change.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
)

query_params = [
    bigquery.ScalarQueryParameter("start_date", "DATE", analysis_start_date),
    bigquery.ScalarQueryParameter("end_date", "DATE", effective_end_date),
    bigquery.ScalarQueryParameter("recent_days", "INT64", recent_days),
    bigquery.ScalarQueryParameter("top_n_pages", "INT64", TOP_N_PAGES),
    bigquery.ScalarQueryParameter("include_homepage", "BOOL", INCLUDE_HOMEPAGE),
    bigquery.ScalarQueryParameter("min_query_impressions", "INT64", MIN_QUERY_IMPRESSIONS),
]

df_query_changes = run_query(client, query_sql, params=query_params)
if df_query_changes.empty:
    raise ValueError("No query-change rows returned. Try lowering MIN_QUERY_IMPRESSIONS.")

print(f"Rows returned: {len(df_query_changes):,}")
df_query_changes.head()

Rows returned: 95,661


,page_path,query,recent_clicks,recent_impressions,recent_avg_position,prior_clicks,prior_impressions,prior_avg_position,click_delta,impression_delta,avg_position_delta,query_state
0,/131114,lifeline,181,15528,3.009080,0,0,NaN,181,15528,3.009080,new_query
1,/131114,life line,16,1756,3.196469,0,0,NaN,16,1756,3.196469,new_query
2,/131114,lifeline number,15,1165,4.283262,0,0,NaN,15,1165,4.283262,new_query
3,/131114,lifeline number australia,12,73,1.273973,0,0,NaN,12,73,1.273973,new_query
4,/131114,lifeline australia,11,1588,3.883501,0,0,NaN,11,1588,3.883501,new_query


## Table 2: Top rising and falling queries

These tables show the strongest positive and negative query movements across your top pages.

Why this helps:
- Identify specific intents/themes driving page gains or losses.
- Distinguish new/lost queries from changes in existing queries.

In [10]:
# Query momentum diagnostics
rising_queries = df_query_changes.sort_values("click_delta", ascending=False).head(30)
falling_queries = df_query_changes.sort_values("click_delta", ascending=True).head(30)

print("Top rising queries across top pages")
display(rising_queries[[
    "page_path",
    "query",
    "query_state",
    "recent_clicks",
    "prior_clicks",
    "click_delta",
    "recent_impressions",
    "prior_impressions",
    "impression_delta",
    "recent_avg_position",
    "prior_avg_position",
    "avg_position_delta",
]])

print("Top falling queries across top pages")
display(falling_queries[[
    "page_path",
    "query",
    "query_state",
    "recent_clicks",
    "prior_clicks",
    "click_delta",
    "recent_impressions",
    "prior_impressions",
    "impression_delta",
    "recent_avg_position",
    "prior_avg_position",
    "avg_position_delta",
]])

Top rising queries across top pages


,page_path,query,query_state,recent_clicks,prior_clicks,click_delta,recent_impressions,prior_impressions,impression_delta,recent_avg_position,prior_avg_position,avg_position_delta
0,/131114,lifeline,new_query,181,0,181,15528,0,15528,3.009080,NaN,3.009080
7305,/chat,online chat,new_query,148,0,148,8557,0,8557,9.429590,NaN,9.429590
3541,/about-us/our-services,lifeline,new_query,120,0,120,13963,0,13963,3.067607,NaN,3.067607
93521,/get-involved/volunteer/volunteer-as-a-crisis-...,lifeline volunteer,new_query,116,0,116,877,0,877,1.746864,NaN,1.746864
72286,/get-help/support-toolkit/techniques-and-guide...,acceptance and commitment therapy,new_query,111,0,111,4978,0,4978,6.497188,NaN,6.497188
82119,/get-help/support-toolkit/tools-and-apps/beyon...,beyond now safety plan,new_query,105,0,105,212,0,212,3.254717,NaN,3.254717
4942,/about/careers,lifeline jobs,new_query,85,0,85,258,0,258,1.934109,NaN,1.934109
5118,/about/contact-us,lifeline,new_query,82,0,82,11406,0,11406,2.801771,NaN,2.801771
7306,/chat,lifeline online chat,new_query,79,0,79,128,0,128,1.257812,NaN,1.257812
7307,/chat,lifeline,new_query,78,0,78,14760,0,14760,3.568360,NaN,3.568360


Top falling queries across top pages


,page_path,query,query_state,recent_clicks,prior_clicks,click_delta,recent_impressions,prior_impressions,impression_delta,recent_avg_position,prior_avg_position,avg_position_delta
62959,/get-help/national-services/national-alcohol-o...,what should i do if i think my neighbors are d...,new_query,0,0,0,1,0,1,1.000000,NaN,1.000000
62970,/get-help/national-services/national-alcohol-o...,drugs stimulant or depressant,new_query,0,0,0,1,0,1,3.000000,NaN,3.000000
62969,/get-help/national-services/national-alcohol-o...,detox for a drug test,new_query,0,0,0,2,0,2,2.000000,NaN,2.000000
62968,/get-help/national-services/national-alcohol-o...,wind addiction,new_query,0,0,0,1,0,1,1.000000,NaN,1.000000
62967,/get-help/national-services/national-alcohol-o...,advice for partners of alcoholics,new_query,0,0,0,1,0,1,3.000000,NaN,3.000000
62966,/get-help/national-services/national-alcohol-o...,arbd symptoms,new_query,0,0,0,2,0,2,2.000000,NaN,2.000000
62965,/get-help/national-services/national-alcohol-o...,st george rehabilitation,new_query,0,0,0,1,0,1,2.000000,NaN,2.000000
62964,/get-help/national-services/national-alcohol-o...,snus withdrawal symptoms,new_query,0,0,0,1,0,1,2.000000,NaN,2.000000
62963,/get-help/national-services/national-alcohol-o...,internet addiction treatment centers,new_query,0,0,0,13,0,13,2.846154,NaN,2.846154
62962,/get-help/national-services/national-alcohol-o...,quit smoking products online,new_query,0,0,0,1,0,1,2.000000,NaN,2.000000


## Table 3: Query change summary by page

This rolls query-level movement up to page level.

Why this helps:
- See which pages have broad-based query momentum vs a few isolated query swings.
- Prioritize pages with high negative `net_query_click_delta` and many falling/lost queries.

In [11]:
# Page-level query change summary
query_summary = (
    df_query_changes.groupby("page_path", as_index=False)
    .agg(
        rising_query_count=("click_delta", lambda s: int((s > 0).sum())),
        falling_query_count=("click_delta", lambda s: int((s < 0).sum())),
        net_query_click_delta=("click_delta", "sum"),
        new_query_count=("query_state", lambda s: int((s == "new_query").sum())),
        lost_query_count=("query_state", lambda s: int((s == "lost_query").sum())),
    )
    .sort_values("net_query_click_delta", ascending=False)
)

display(query_summary.head(20))

,page_path,rising_query_count,falling_query_count,net_query_click_delta,new_query_count,lost_query_count
11,/get-help/national-services/national-alcohol-o...,1745,0,2550,52536,0
5,/chat,312,0,1031,7576,0
21,/get-help/support-toolkit/topics/self-harm,276,0,419,5205,0
23,/get-involved/volunteer/volunteer-as-a-crisis-...,91,0,337,658,0
0,/131114,69,0,324,3541,0
12,/get-help/support-toolkit/safety-planning/beyo...,49,0,264,464,0
19,/get-help/support-toolkit/tools-and-apps/i-am-...,97,0,259,1096,0
13,/get-help/support-toolkit/techniques-and-guide...,38,0,239,1074,0
1,/about-us/our-services,46,0,236,1401,0
2,/about/careers,46,0,224,176,0


## Chart 3: SXO category impact view

This groups pages into broad SXO categories and compares net movement.

Why this helps:
- Move from page-level noise to strategic pattern detection.
- Spot whether changes are concentrated in crisis support, self-led resources, fundraising, or other areas.

In [12]:
# Optional SXO category lens (simple prefix mapping)
def classify_sxo_category(path: str) -> str:
    if path is None:
        return "about_us_or_other"
    if path.startswith("/get-help/support-toolkit") or path.startswith("/get-help/national-services") or path.startswith("/get-help/hear-from-others"):
        return "self_led_support"
    if path in {
        "/chat",
        "/131114",
        "/text",
        "/get-help/im-thinking-about-suicide",
        "/get-help/i-want-to-talk-to-someone",
        "/get-help/i-want-to-explore-my-options",
        "/get-help/services/crisis-support",
    }:
        return "support_services"
    if path.startswith("/get-involved/volunteer"):
        return "volunteering"
    if (
        path.startswith("/get-involved/fundraising")
        or path.startswith("/get-involved/donate")
        or path == "/about/our-partners/our-corporate-partners"
        or path == "/get-involved/partner-with-us"
    ):
        return "marketing_fundraising"
    return "about_us_or_other"

page_movers["sxo_category"] = page_movers["page_path"].map(classify_sxo_category)
category_summary = (
    page_movers.groupby("sxo_category", as_index=False)
    .agg(
        pages=("page_path", "nunique"),
        recent_clicks=("recent_clicks", "sum"),
        prior_clicks=("prior_clicks", "sum"),
        net_click_delta=("click_delta", "sum"),
        recent_impressions=("recent_impressions", "sum"),
        prior_impressions=("prior_impressions", "sum"),
        net_impression_delta=("impression_delta", "sum"),
    )
    .sort_values("recent_clicks", ascending=False)
)

display(category_summary)

fig_category = px.bar(
    category_summary,
    x="sxo_category",
    y="net_click_delta",
    color="net_impression_delta",
    template="lifeline",
    title="Net click change by SXO category (recent vs prior)",
)
lifeline_theme.add_lifeline_logo(fig_category)
fig_category.show()

,sxo_category,pages,recent_clicks,prior_clicks,net_click_delta,recent_impressions,prior_impressions,net_impression_delta
2,self_led_support,15,12393,0,12393,1152580,0,1152580
3,support_services,3,2049,0,2049,426453,0,426453
0,about_us_or_other,5,1642,0,1642,150132,0,150132
4,volunteering,1,713,0,713,16314,0,16314
1,marketing_fundraising,1,279,0,279,25346,0,25346


## SXO Interpretation Guide

Use the outputs above to classify pattern shifts before actioning:

- **Help-seeking demand shift**: impressions rise across crisis-related pages and related urgent queries.
- **Discoverability shift**: clicks drop while impressions hold, and average position worsens.
- **SERP competitiveness shift**: many continuing queries show worse position with stable intent demand.

## Practical action checklist
- For crisis-support pages with positive demand shifts, validate service-path prominence and capacity signals.
- For self-led support pages with impression growth but weak CTR, improve snippets/meta + on-page alignment to query intent.
- For volunteer/fundraising pages with rising demand, verify conversion path clarity and measurement.
- For pages with strong losses, inspect query-level losses to find specific intent gaps to address first.

Tip: Keep `INCLUDE_HOMEPAGE=False` for strategic diagnosis; toggle to `True` when you need full-site context.